# Exponential-filter RZSM prediction and evaluation

This notebook follows the same execution order as the older `Exponential_Filter` scripts:

1. `01_EF_calculation.py`: export the fixed-T=15 ASCAT and SMAP RZSM series, validate each run against the canonical FNO point cohort, and write station metrics.
2. `02_EF_scatterplot.py`: evaluate pooled test-period predictions and draw density scatterplots.
3. `03_EF_timeseries_plot.py`: write full-period and test-period point-metric CSV files first, then draw timeseries figures in a separate cell that consumes those CSV files.

The fixed-T series are produced by `ISMN_preprocessing.ipynb`; this workflow evaluates and exports them without recalculating or optimizing T. The recursive filter advances only on valid SSM observation dates, leaves other dates missing, and publishes RZSM after the one-calendar-year adjustment ending 2016-03-31. Reusable functions are stored in `EF/`, while paths, scientific settings, loops, and execution calls remain visible here. After the metric-CSV cell finishes, the plotting cell can be run or rerun independently.

In [1]:
import glob
import multiprocessing as mp
import os

from config import configure_runtime

configure_runtime()

import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm

## 1. Paths and reusable imports

In [2]:
from config import (
    EF_WORKERS,
    base_FP,
    cpuserver_data,
    das_FP,
    george_FP,
    nas_FP,
)

import HydroAI.Grid as hGrid

from EF.evaluation import (
    evaluate_pixel_task,
    evaluate_timeseries_file,
    select_timeseries_file,
)
from EF.integrity import (
    atomic_write_csv,
    clean_matching_files,
)
from EF.plotting import plot_timeseries, scatterplot
from EF.prediction import (
    SUMMARY_COLUMNS,
    canonical_pixel_files,
    prediction_run_status,
    process_station_file,
)

from config import figures_FP, results_FP


## 2. Scientific and runtime settings

In [3]:
DATA_SOURCES = ("ASCAT", "SMAP")
PREDICTION_AREAS = ("Train", "Test", "Excluded")
EVALUATION_AREAS = ("Train", "Test", "Excluded")

TARGET_VARIABLE = "in-situ_RZSM"
EF_VARIABLES = {
    "ASCAT": "ASCAT_RZSM",
    "SMAP": "SMAP_RZSM",
}
FILTER_TIME_DAYS = 15.0
EF_INPUT_START = pd.Timestamp("2015-04-01")
EF_INPUT_END = pd.Timestamp("2023-12-31")
EXPECTED_INPUT_DATES = pd.date_range(EF_INPUT_START, EF_INPUT_END, freq="D")
EXPECTED_INPUT_DAYS = 3_197
if len(EXPECTED_INPUT_DATES) != EXPECTED_INPUT_DAYS:
    raise RuntimeError(f"Unexpected EF input period: {len(EXPECTED_INPUT_DATES):,} days")
EF_OUTPUT_START = pd.Timestamp("2016-03-31")
SPIN_UP_DAYS = float((EF_OUTPUT_START - EF_INPUT_START).days)
GRID_RESOLUTION_DEGREES = 0.10

TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.10
TEST_START_RATIO_AFTER_SPINUP = round(TRAIN_RATIO + VALIDATION_RATIO, 2)
TEST_PERIOD_POLICY = "final_20_percent_of_post_spinup_common_period"
TEST_PERIOD_END = EF_INPUT_END
post_spinup_dates = pd.date_range(EF_OUTPUT_START, TEST_PERIOD_END, freq="D")
test_start_index = min(
    int(len(post_spinup_dates) * TEST_START_RATIO_AFTER_SPINUP),
    len(post_spinup_dates) - 1,
)
TEST_PERIOD_START = post_spinup_dates[test_start_index]
TEST_PERIOD_METADATA = {
    "test_period_policy": TEST_PERIOD_POLICY,
    "test_start_ratio_after_spinup": TEST_START_RATIO_AFTER_SPINUP,
    "test_start_date": TEST_PERIOD_START.strftime("%Y-%m-%d"),
    "test_end_date": TEST_PERIOD_END.strftime("%Y-%m-%d"),
}
WORKERS = EF_WORKERS

SCATTER_AXIS_LIMITS = (0.0, 0.55)
SCATTER_BINS = 100
SCATTER_DENSITY_MAXIMUM = 100

TIMESERIES_PERIODS = {
    "Full": {
        "start_date": None,
        "end_date": None,
        "x_limits": ("2015-04-01", "2023-12-31"),
        "tick_dates": [f"{year}-01-01" for year in range(2016, 2024)],
        "tick_format": "%Y",
        "figure_size": (12, 6),
        "figure_folder": "Full_timeseries",
        "csv_suffix": "full",
    },
    "Test": {
        "start_date": TEST_PERIOD_START,
        "end_date": TEST_PERIOD_END,
        "x_limits": None,
        "tick_dates": [
            "2022-04-01", "2022-07-01", "2022-10-01",
            "2023-01-01", "2023-04-01", "2023-07-01",
            "2023-10-01", "2023-12-31",
        ],
        "tick_format": "%Y-%m-%d",
        "figure_size": (12, 8),
        "figure_folder": "Test_period",
        "csv_suffix": "test",
    },
}

In [4]:
ISMN_RESULT_FP = os.path.join(results_FP, "ISMN"
)
FNO_RESULT_FP = os.path.join(results_FP, "FNO")
EF_RESULT_FP = os.path.join(results_FP, "EF")
PREDICTION_ROOT_FP = os.path.join(EF_RESULT_FP, "RZSM_prediction")
COMPARISON_ROOT_FP = os.path.join(EF_RESULT_FP, "RZSM_comparison")
SCATTER_FIGURE_FP = os.path.join(figures_FP, "EF", "Scatterplot"
)
TIMESERIES_FIGURE_FP = os.path.join(figures_FP, "EF", "Timeseries_comparison"
)

for directory in (
    PREDICTION_ROOT_FP,
    COMPARISON_ROOT_FP,
    SCATTER_FIGURE_FP,
    TIMESERIES_FIGURE_FP,
):
    os.makedirs(directory, exist_ok=True)

eqd_longitude, eqd_latitude = hGrid.generate_lon_lat_eqdgrid(
    GRID_RESOLUTION_DEGREES
)
LONGITUDE_AXIS = eqd_longitude[0, :]
LATITUDE_AXIS = eqd_latitude[:, 0]

print(f"EF workers: {WORKERS}")
print(f"Grid shape: {eqd_longitude.shape}")

EF workers: 60
Grid shape: (1800, 3600)


## 3. Legacy step 1: export fixed-T predictions and station metrics

Each output contains the original SSM and observed RZSM plus the corresponding embedded ASCAT or SMAP fixed-T RZSM series as `RZSM_prediction`. Existing run-specific point files are replaced only within the current area/product output folder.

In [5]:
prediction_summaries = {}

for area in PREDICTION_AREAS:
    for data_type in DATA_SOURCES:
        input_directory = os.path.join(
            ISMN_RESULT_FP, f"ISMN_{area}", data_type
        )
        canonical_csv = os.path.join(
            FNO_RESULT_FP, area, f"FNO_input_{data_type}.csv"
        )
        filenames = canonical_pixel_files(
            canonical_csv,
            input_directory,
            expected_start=EF_INPUT_START,
            expected_end=EF_INPUT_END,
            expected_count=EXPECTED_INPUT_DAYS,
        )

        area_output = os.path.join(PREDICTION_ROOT_FP, area)
        prediction_directory = os.path.join(area_output, data_type)
        os.makedirs(prediction_directory, exist_ok=True)

        removed_files = []
        removed_files.extend(
            clean_matching_files(prediction_directory, "*_prediction.nc")
        )
        removed_files.extend(
            clean_matching_files(prediction_directory, "*_prediction.nc.tmp.*")
        )

        tasks = [
            {
                "filename": filename,
                "input_directory": input_directory,
                "output_directory": prediction_directory,
                "latitude_axis": LATITUDE_AXIS,
                "longitude_axis": LONGITUDE_AXIS,
                "data_type": data_type,
                "target_variable": TARGET_VARIABLE,
                "ef_variables": EF_VARIABLES,
                "filter_time_days": FILTER_TIME_DAYS,
                "spin_up_days": SPIN_UP_DAYS,
            }
            for filename in filenames
        ]

        with mp.Pool(processes=WORKERS) as pool:
            results = list(
                tqdm(
                    pool.imap(process_station_file, tasks),
                    total=len(tasks),
                    desc=f"Exporting {area}-{data_type}",
                    leave=False,
                )
            )

        status = prediction_run_status(
            filenames, results, prediction_directory
        )
        if status["missing_outputs"]:
            preview = ", ".join(status["missing_outputs"][:10])
            if len(status["missing_outputs"]) > 10:
                preview += ", ..."
            raise RuntimeError(
                f"Incomplete EF run for {area} {data_type}: "
                f"{len(status['missing_outputs'])} outputs are missing "
                f"({preview})"
            )

        summary_df = pd.DataFrame(results, columns=SUMMARY_COLUMNS)
        summary_df = summary_df.sort_values(["lat_idx", "lon_idx"])
        summary_file = os.path.join(area_output, f"T_cal_{data_type}.csv")
        atomic_write_csv(summary_df, summary_file)
        prediction_summaries[(area, data_type)] = {
            "pixels": len(summary_df),
            "warnings": len(status["warning_pixels"]),
            "replaced_files": len(removed_files),
            "summary_file": summary_file,
        }

display(pd.DataFrame(prediction_summaries).T)

Exporting Train-ASCAT:   0%|          | 0/105 [00:00<?, ?it/s]

Exporting Train-SMAP:   0%|          | 0/105 [00:00<?, ?it/s]

Exporting Test-ASCAT:   0%|          | 0/79 [00:00<?, ?it/s]

Exporting Test-SMAP:   0%|          | 0/79 [00:00<?, ?it/s]

Exporting Excluded-ASCAT:   0%|          | 0/109 [00:00<?, ?it/s]

Exporting Excluded-SMAP:   0%|          | 0/94 [00:00<?, ?it/s]

pixels warnings replaced_files  \
Train    ASCAT    105        0            105   
         SMAP     105        0            105   
Test     ASCAT     79        0             79   
         SMAP      79        0             79   
Excluded ASCAT    109        0            109   
         SMAP      94        0             94   

                                                     summary_file  
Train    ASCAT  /home/kunhee/cpuserver_data/python_modules/kun...  
         SMAP   /home/kunhee/cpuserver_data/python_modules/kun...  
Test     ASCAT  /home/kunhee/cpuserver_data/python_modules/kun...  
         SMAP   /home/kunhee/cpuserver_data/python_modules/kun...  
Excluded ASCAT  /home/kunhee/cpuserver_data/python_modules/kun...  
         SMAP   /home/kunhee/cpuserver_data/python_modules/kun...

## 4. Legacy step 2: pooled test-period density scatterplots

The evaluation period is the final 20% of the post-spin-up common daily record: 2022-06-13 through 2023-12-31, inclusive. Every ISMN point input must first match the exact 3,197-day study coordinate from 2015-04-01 through 2023-12-31. Only finite observed/predicted pairs are pooled.

In [6]:
scatter_summaries = {}

for area in EVALUATION_AREAS:
    for data_type in tqdm(
        DATA_SOURCES, desc=f"{area} scatterplots", leave=False
    ):
        area_output = os.path.join(PREDICTION_ROOT_FP, area)
        prediction_directory = os.path.join(area_output, data_type)
        summary_file = os.path.join(area_output, f"T_cal_{data_type}.csv")
        if not os.path.exists(summary_file):
            print(f"Skipping {area}-{data_type}: missing {summary_file}")
            continue
        summary_df = pd.read_csv(summary_file)
        tasks = [
            {
                "lat_idx": row.lat_idx,
                "lon_idx": row.lon_idx,
                "prediction_directory": prediction_directory,
                "start_date": TEST_PERIOD_START,
                "end_date": TEST_PERIOD_END,
            }
            for row in summary_df.itertuples()
        ]

        with mp.Pool(processes=WORKERS) as pool:
            pixel_results = list(
                pool.imap_unordered(evaluate_pixel_task, tasks)
            )
        pixel_results = [result for result in pixel_results if result is not None]
        all_observed = (
            np.concatenate([result["observed"] for result in pixel_results])
            if pixel_results else np.array([])
        )
        all_predicted = (
            np.concatenate([result["predicted"] for result in pixel_results])
            if pixel_results else np.array([])
        )
        output_file = os.path.join(
            SCATTER_FIGURE_FP, f"EF_{data_type}_{area}_comparison.png"
        )
        metrics = scatterplot(
            observed=all_observed,
            predicted=all_predicted,
            pixel_count=len(pixel_results),
            data_type=data_type,
            area=area,
            output_file=output_file,
            axis_limits=SCATTER_AXIS_LIMITS,
            bins=SCATTER_BINS,
            density_maximum=SCATTER_DENSITY_MAXIMUM,
        )
        if metrics is None:
            print(f"No scatterplot written for {area}-{data_type}: too few pairs")
            continue
        scatter_summaries[(area, data_type)] = {
            "pixels": len(pixel_results),
            **metrics,
            "figure": output_file,
        }

display(pd.DataFrame(scatter_summaries).T)

Train scatterplots:   0%|          | 0/2 [00:00<?, ?it/s]

Test scatterplots:   0%|          | 0/2 [00:00<?, ?it/s]

Excluded scatterplots:   0%|          | 0/2 [00:00<?, ?it/s]

pixels         R      RMSE      Bias    ubRMSE      N  \
Train    ASCAT    103  0.641345  0.092868 -0.042817  0.082408  40353   
         SMAP     103  0.654096  0.082912 -0.015001  0.081543  20743   
Test     ASCAT     77  0.556921  0.088885 -0.030899  0.083341  30858   
         SMAP      77  0.462713  0.098523  0.004315  0.098429  15681   
Excluded ASCAT    109  0.467584   0.11897 -0.050158   0.10788  37657   
         SMAP      94  0.554585  0.102231   -0.0063  0.102036  17271   

                                                           figure  
Train    ASCAT  /home/kunhee/cpuserver_data/python_modules/kun...  
         SMAP   /home/kunhee/cpuserver_data/python_modules/kun...  
Test     ASCAT  /home/kunhee/cpuserver_data/python_modules/kun...  
         SMAP   /home/kunhee/cpuserver_data/python_modules/kun...  
Excluded ASCAT  /home/kunhee/cpuserver_data/python_modules/kun...  
         SMAP   /home/kunhee/cpuserver_data/python_modules/kun...

## 5. Legacy step 3: full-period and test-period point timeseries

Full-period metrics use the exact 3,197-day coordinate from 2015-04-01 through 2023-12-31 in each point file. Test metrics and figures use the final-20% window from 2022-06-13 through 2023-12-31, inclusive, matching the pooled scatterplots.

### 5.1 Calculate and save metric CSV files

Run this cell first. It calculates both full-period and test-period metrics and writes each CSV atomically. Test-period CSVs include the period policy, ratio, and dates. It does not render figures, so the CSV files are available before the slower plotting phase begins.

In [7]:
timeseries_metric_summaries = []

for period_name, period_settings in TIMESERIES_PERIODS.items():
    for area in EVALUATION_AREAS:
        for data_type in DATA_SOURCES:
            comparison_directory = os.path.join(COMPARISON_ROOT_FP, area)
            prediction_directory = os.path.join(
                PREDICTION_ROOT_FP, area, data_type
            )
            os.makedirs(comparison_directory, exist_ok=True)

            prediction_files = sorted(
                glob.glob(
                    os.path.join(prediction_directory, "*_prediction.nc")
                )
            )
            metric_rows = []
            for prediction_file in tqdm(
                prediction_files,
                desc=f"{period_name} {area}-{data_type}",
                leave=False,
            ):
                filename = os.path.basename(prediction_file)
                latitude_index, longitude_index = map(
                    int, filename.split("_")[:2]
                )
                latitude = round(float(LATITUDE_AXIS[latitude_index]), 2)
                longitude_west = abs(
                    round(float(LONGITUDE_AXIS[longitude_index]), 2)
                )
                _, metrics = evaluate_timeseries_file(
                    prediction_file,
                    start_date=period_settings["start_date"],
                    end_date=period_settings["end_date"],
                )

                metric_row = {
                    "lat_idx": latitude_index,
                    "lon_idx": longitude_index,
                }
                if period_name == "Test":
                    metric_row.update(TEST_PERIOD_METADATA)
                    metric_row.update({"lat": latitude, "lon": longitude_west})
                metric_row.update(
                    {
                        "R": metrics["R"],
                        "RMSE": metrics["RMSE"],
                        "bias": metrics["Bias"],
                        "ubRMSE": metrics["ubRMSE"],
                    }
                )
                if period_name == "Test":
                    metric_row["N"] = metrics["N"]
                metric_rows.append(metric_row)

            if metric_rows:
                metrics_df = pd.DataFrame(metric_rows).sort_values(
                    ["lat_idx", "lon_idx"]
                )
                metrics_file = os.path.join(
                    comparison_directory,
                    f"{data_type}_{period_settings['csv_suffix']}_comparison.csv",
                )
                atomic_write_csv(metrics_df, metrics_file)
                timeseries_metric_summaries.append(
                    {
                        "period": period_name,
                        "area": area,
                        "data": data_type,
                        "pixels": len(metrics_df),
                        "metrics_file": metrics_file,
                    }
                )

display(pd.DataFrame(timeseries_metric_summaries))

Full Train-ASCAT:   0%|          | 0/105 [00:00<?, ?it/s]

Full Train-SMAP:   0%|          | 0/105 [00:00<?, ?it/s]

Full Test-ASCAT:   0%|          | 0/79 [00:00<?, ?it/s]

Full Test-SMAP:   0%|          | 0/79 [00:00<?, ?it/s]

Full Excluded-ASCAT:   0%|          | 0/109 [00:00<?, ?it/s]

...

Test Test-ASCAT:   0%|          | 0/79 [00:00<?, ?it/s]

Test Test-SMAP:   0%|          | 0/79 [00:00<?, ?it/s]

Test Excluded-ASCAT:   0%|          | 0/109 [00:00<?, ?it/s]

Test Excluded-SMAP:   0%|          | 0/94 [00:00<?, ?it/s]

,period,area,data,pixels,metrics_file
0,Full,Train,ASCAT,105,/home/kunhee/cpuserver_data/python_modules/kun...
1,Full,Train,SMAP,105,/home/kunhee/cpuserver_data/python_modules/kun...
2,Full,Test,ASCAT,79,/home/kunhee/cpuserver_data/python_modules/kun...
3,Full,Test,SMAP,79,/home/kunhee/cpuserver_data/python_modules/kun...
4,Full,Excluded,ASCAT,109,/home/kunhee/cpuserver_data/python_modules/kun...
5,Full,Excluded,SMAP,94,/home/kunhee/cpuserver_data/python_modules/kun...
6,Test,Train,ASCAT,105,/home/kunhee/cpuserver_data/python_modules/kun...
7,Test,Train,SMAP,105,/home/kunhee/cpuserver_data/python_modules/kun...
8,Test,Test,ASCAT,79,/home/kunhee/cpuserver_data/python_modules/kun...
9,Test,Test,SMAP,79,/home/kunhee/cpuserver_data/python_modules/kun...


### 5.2 Plot figures from the saved metric CSV files

Run this cell after Section 5.1. It does not recalculate or overwrite the metric CSV files. Before plotting, it verifies that each CSV contains exactly the prediction pixels.

In [8]:
timeseries_plot_summaries = []

for period_name, period_settings in TIMESERIES_PERIODS.items():
    for area in EVALUATION_AREAS:
        for data_type in DATA_SOURCES:
            comparison_directory = os.path.join(COMPARISON_ROOT_FP, area)
            prediction_directory = os.path.join(
                PREDICTION_ROOT_FP, area, data_type
            )
            figure_directory = os.path.join(
                TIMESERIES_FIGURE_FP,
                period_settings["figure_folder"],
                area,
                data_type,
            )
            os.makedirs(figure_directory, exist_ok=True)

            prediction_files = sorted(
                glob.glob(
                    os.path.join(prediction_directory, "*_prediction.nc")
                )
            )
            metrics_file = os.path.join(
                comparison_directory,
                f"{data_type}_{period_settings['csv_suffix']}_comparison.csv",
            )
            if not os.path.exists(metrics_file):
                raise FileNotFoundError(
                    f"Missing metric CSV: {metrics_file}. Run Section 5.1 first."
                )
            metrics_df = pd.read_csv(metrics_file)
            required_columns = {
                "lat_idx",
                "lon_idx",
                "R",
                "RMSE",
                "bias",
                "ubRMSE",
            }
            if period_name == "Test":
                required_columns.update(TEST_PERIOD_METADATA)
                required_columns.update({"lat", "lon", "N"})
            missing_columns = required_columns - set(metrics_df.columns)
            if missing_columns:
                raise KeyError(
                    f"{metrics_file} is missing columns: {sorted(missing_columns)}"
                )
            if metrics_df.duplicated(["lat_idx", "lon_idx"]).any():
                raise RuntimeError(f"Duplicate pixel keys in {metrics_file}")
            if period_name == "Test":
                for name, expected_value in TEST_PERIOD_METADATA.items():
                    values = metrics_df[name].dropna().unique()
                    if isinstance(expected_value, float):
                        matches = (
                            len(values) == 1
                            and np.isclose(float(values[0]), expected_value)
                        )
                    else:
                        matches = (
                            len(values) == 1
                            and str(values[0]) == str(expected_value)
                        )
                    if not matches:
                        raise RuntimeError(
                            f"Stale or mixed {name} in {metrics_file}: "
                            f"{values.tolist()}; expected {expected_value!r}"
                        )

            expected_pixels = {
                tuple(map(int, os.path.basename(path).split("_")[:2]))
                for path in prediction_files
            }
            csv_pixels = {
                (int(row.lat_idx), int(row.lon_idx))
                for row in metrics_df.itertuples()
            }
            if csv_pixels != expected_pixels:
                raise RuntimeError(
                    f"Metric CSV pixel cohort does not match {prediction_directory}"
                )
            metrics_by_pixel = metrics_df.set_index(["lat_idx", "lon_idx"])

            for prediction_file in tqdm(
                prediction_files,
                desc=f"Plotting {period_name} {area}-{data_type}",
                leave=False,
            ):
                filename = os.path.basename(prediction_file)
                latitude_index, longitude_index = map(
                    int, filename.split("_")[:2]
                )
                latitude = round(float(LATITUDE_AXIS[latitude_index]), 2)
                longitude_west = abs(
                    round(float(LONGITUDE_AXIS[longitude_index]), 2)
                )
                series = select_timeseries_file(
                    prediction_file,
                    start_date=period_settings["start_date"],
                    end_date=period_settings["end_date"],
                )
                metric_row = metrics_by_pixel.loc[
                    (latitude_index, longitude_index)
                ]
                metrics = {
                    "R": metric_row["R"],
                    "RMSE": metric_row["RMSE"],
                    "Bias": metric_row["bias"],
                    "ubRMSE": metric_row["ubRMSE"],
                }

                plot_timeseries(
                    dates=series["time"],
                    surface_ssm=series["SSM"],
                    observed_rzsm=series["RZSM"],
                    predicted_rzsm=series["RZSM_prediction"],
                    metrics=metrics,
                    latitude=latitude,
                    longitude_west=longitude_west,
                    output_file=os.path.join(
                        figure_directory,
                        f"{latitude_index}_{longitude_index}_RZSM_comparison.png",
                    ),
                    figure_size=period_settings["figure_size"],
                    tick_dates=period_settings["tick_dates"],
                    tick_format=period_settings["tick_format"],
                    x_limits=period_settings["x_limits"],
                )

            timeseries_plot_summaries.append(
                {
                    "period": period_name,
                    "area": area,
                    "data": data_type,
                    "figures": len(prediction_files),
                    "metrics_file": metrics_file,
                    "figure_directory": figure_directory,
                }
            )

display(pd.DataFrame(timeseries_plot_summaries))

Plotting Full Train-ASCAT:   0%|          | 0/105 [00:00<?, ?it/s]

Plotting Full Train-SMAP:   0%|          | 0/105 [00:00<?, ?it/s]

Plotting Full Test-ASCAT:   0%|          | 0/79 [00:00<?, ?it/s]

Plotting Full Test-SMAP:   0%|          | 0/79 [00:00<?, ?it/s]

Plotting Full Excluded-ASCAT:   0%|          | 0/109 [00:00<?, ?it/s]

...

Plotting Test Test-ASCAT:   0%|          | 0/79 [00:00<?, ?it/s]

Plotting Test Test-SMAP:   0%|          | 0/79 [00:00<?, ?it/s]

Plotting Test Excluded-ASCAT:   0%|          | 0/109 [00:00<?, ?it/s]

Plotting Test Excluded-SMAP:   0%|          | 0/94 [00:00<?, ?it/s]

,period,area,data,figures,metrics_file,figure_directory
0,Full,Train,ASCAT,105,/home/kunhee/cpuserver_data/python_modules/kun...,/home/kunhee/cpuserver_data/python_modules/kun...
1,Full,Train,SMAP,105,/home/kunhee/cpuserver_data/python_modules/kun...,/home/kunhee/cpuserver_data/python_modules/kun...
2,Full,Test,ASCAT,79,/home/kunhee/cpuserver_data/python_modules/kun...,/home/kunhee/cpuserver_data/python_modules/kun...
3,Full,Test,SMAP,79,/home/kunhee/cpuserver_data/python_modules/kun...,/home/kunhee/cpuserver_data/python_modules/kun...
4,Full,Excluded,ASCAT,109,/home/kunhee/cpuserver_data/python_modules/kun...,/home/kunhee/cpuserver_data/python_modules/kun...
5,Full,Excluded,SMAP,94,/home/kunhee/cpuserver_data/python_modules/kun...,/home/kunhee/cpuserver_data/python_modules/kun...
6,Test,Train,ASCAT,105,/home/kunhee/cpuserver_data/python_modules/kun...,/home/kunhee/cpuserver_data/python_modules/kun...
7,Test,Train,SMAP,105,/home/kunhee/cpuserver_data/python_modules/kun...,/home/kunhee/cpuserver_data/python_modules/kun...
8,Test,Test,ASCAT,79,/home/kunhee/cpuserver_data/python_modules/kun...,/home/kunhee/cpuserver_data/python_modules/kun...
9,Test,Test,SMAP,79,/home/kunhee/cpuserver_data/python_modules/kun...,/home/kunhee/cpuserver_data/python_modules/kun...


## 6. Output inspection

In [9]:
sample_summary = os.path.join(PREDICTION_ROOT_FP, "Train", "T_cal_ASCAT.csv")
sample_prediction = sorted(
    glob.glob(os.path.join(PREDICTION_ROOT_FP, "Train", "ASCAT", "*_prediction.nc"))
)

if os.path.exists(sample_summary):
    display(pd.read_csv(sample_summary).head())
if sample_prediction:
    with xr.open_dataset(sample_prediction[0]) as sample_dataset:
        display(sample_dataset)

print(f"Results folder: {EF_RESULT_FP}")
print(f"Figures folder: {os.path.dirname(SCATTER_FIGURE_FP)}")

,lat_idx,lon_idx,lat,lon,R,RMSE,KGE,Bias,Flag
0,415,687,48.45,-111.25,0.7305,0.0812,0.5525,-0.0627,NaN
1,421,757,47.85,-104.25,0.8094,0.0866,0.5660,-0.0791,NaN
2,422,836,47.75,-96.35,0.6980,0.0701,0.6153,0.0427,NaN
3,424,728,47.55,-107.15,0.7622,0.1418,0.4288,-0.1345,NaN
4,428,699,47.15,-110.05,0.7968,0.1141,0.5277,-0.1077,NaN


<xarray.Dataset> Size: 77kB
Dimensions:          (time: 3197)
Coordinates:
  * time             (time) datetime64[ns] 26kB 2015-04-01 ... 2023-12-31
Data variables:
    SSM              (time) float32 13kB ...
    RZSM             (time) float32 13kB ...
    RZSM_prediction  (time) float64 26kB ...
Attributes:
    source_point_file:                   /home/kunhee/cpuserver_data/python_m...
    source_prediction_variable:          ASCAT_RZSM
    target_variable:                     in-situ_RZSM
    filter_T_days:                       15.0
    adjustment_period:                   one year
    adjustment_days_for_fixed_calendar:  365.0
    study_period_start:                  2015-04-01
    study_period_end:                    2023-12-31
    study_period_days:                   3197
    output_support:                      valid ASCAT SSM observation dates only

Results folder: /home/kunhee/cpuserver_data/python_modules/kunhee/RZSM_FNO_code/Results/EF
Figures folder: /home/kunhee/cpuserver_data/python_modules/kunhee/RZSM_FNO_code/Outputs/Figures/EF
